# Module 2 · Lesson 04: The PCTF Framework

**PCTF = Persona · Context · Task · Format**

A systematic approach to writing effective prompts. Instead of guessing,
use this framework to structure every prompt.

## What you will learn
1. The four PCTF components
2. Real-world examples (code review, documentation)
3. Reusable **prompt templates**
4. Adding **negative constraints**
5. Building an interactive PCTF builder

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0.7, max_tokens=500):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
print("✅ Ready")

✅ Ready


---
## The PCTF Framework

| Component | Question | Example |
|-----------|----------|---------|
| **P**ersona | Who should the AI be? | "You are a senior code reviewer" |
| **C**ontext | What's the situation? | "We're building a FastAPI backend" |
| **T**ask | What should it do? | "Review this function for bugs" |
| **F**ormat | How should it respond? | "Use a numbered list with severity" |

---
## 1. Example: Code Review

In [2]:
system_prompt = """You are senior Python developer with 10 years of experience.
You specialize in code quality, security and performance.
"""
 
user_prompt = """Context: We' re building a REST API with FastAPI. This function handles user login.
 
Task: Review this code for bugs, security issues and performance problems.
 
```python
def login(username, password):
    user = db.query(f"SELECT * FROM users WHERE username='{username}'")
    if user and user.password == password:
        token = str(random.randint(1000, 9999))
        return {"token": token}
    return {"error": "Invalid credentials"}
```
 
Format: For each issue found, provide:
1. Issue severity (Critical, Warning, Suggestion)
2. Line reference
3. Problem description
4. Corrected code
 
"""

result = ask(user_prompt, system=system_prompt)
display(Markdown(result))

Here's a review of the provided `login` function with identified issues related to bugs, security, and performance:

### 1. SQL Injection Vulnerability
- **Severity**: Critical
- **Line Reference**: `user = db.query(f"SELECT * FROM users WHERE username='{username}'")`
- **Problem Description**: The use of string interpolation in SQL queries (`f"SELECT * FROM users WHERE username='{username}'"`) exposes the application to SQL injection attacks. An attacker could manipulate the `username` input to execute arbitrary SQL commands.
- **Corrected Code**:
```python
from sqlalchemy import select

def login(username, password):
    user = db.execute(select(User).where(User.username == username)).scalars().first()
```

### 2. Password Storage and Comparison
- **Severity**: Critical
- **Line Reference**: `if user and user.password == password:`
- **Problem Description**: Passwords should never be stored in plaintext. Instead, they should be hashed using a secure hashing algorithm (e.g., bcrypt) and compared using a secure method. This line performs a direct comparison, which is insecure.
- **Corrected Code**:
```python
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def login(username, password):
    user = db.execute(select(User).where(User.username == username)).scalars().first()
    if user and pwd_context.verify(password, user.password):
```

### 3. Token Generation
- **Severity**: Suggestion
- **Line Reference**: `token = str(random.randint(1000, 9999))`
- **Problem Description**: Using `random.randint` for token generation is insecure as it may produce predictable results. Tokens should be generated using a secure method.
- **Corrected Code**:
```python
import secrets

def login(username, password):
    user = db.execute(select(User).where(User.username == username)).scalars().first()
    if user and pwd_context.verify(password, user.password):
        token = secrets.token_hex(16)  # Generates a secure random token
```

### 4. Lack of User Feedback
- **Severity**: Suggestion
- **Line Reference**: `return {"error": "Invalid credentials"}`
- **Problem Description**: The error message "Invalid credentials" does not specify whether the username or password

---
## 2. Example: API Documentation

In [3]:
system_prompt = """You are a technical writer specializing in API documentation.
Write clear, concise docs that developers can use immediately.
"""
 
user_prompt = """Context: We have a user management API built with FastAPI.
 
Task: Write API documentation for this endpoint.
 
```python
@app.post("/users")
def create_user(name: str, email: str, role: str = "viewer"):
    user = User(name=name, email=email, role=role)
    db.add(user)
    return {"id": user.id, "name": user.name}
```
 
Format: Include:
- Endpoint summary
- Parameters table (name, type, required, description)
- Example request (curl)
- Example response (JSON)
- Possible error codes"""
 
result = ask(user_prompt, system=system_prompt, max_tokens=1600)
display(Markdown(result))
 

# User Management API Documentation

## Endpoint Summary
**POST /users**  
Create a new user in the system. By default, new users are assigned the role of "viewer".

### Parameters

| Name  | Type   | Required | Description                         |
|-------|--------|----------|-------------------------------------|
| name  | string | Yes      | The name of the user.              |
| email | string | Yes      | The email address of the user.     |
| role  | string | No       | The role of the user (default: "viewer"). Possible values: "admin", "editor", "viewer". |

### Example Request
```bash
curl -X POST "http://yourapi.com/users" \
-H "Content-Type: application/json" \
-d '{
  "name": "John Doe",
  "email": "john.doe@example.com",
  "role": "editor"
}'
```

### Example Response
```json
{
  "id": 123,
  "name": "John Doe"
}
```

### Possible Error Codes
- **400 Bad Request**: The request was invalid. This typically occurs when required parameters are missing or invalid.
- **409 Conflict**: A user with the specified email already exists.
- **500 Internal Server Error**: An unexpected error occurred on the server while processing the request.

---
## 3. Reusable Prompt Templates

In production, you build **template functions** for common tasks:

In [4]:
def create_pctf_prompt(persona: str, context: str, task: str, format_spec: str) -> str:
    """Build a structure PCTF prompt"""
    return f"""
    ##Persona
    {persona}
    ### Context
    {context}

    ## Task
    {task}

    ## Format
    {format_spec}"""

In [5]:
prompt = create_pctf_prompt(
    persona="You are a database expert with deep knowledge of SQL optimization.",
    context="We have a PostgreSQL database with 10M rows in the `orders` table. Queries are slow.",
    task="Suggest 3 specific optimizations for this query: SELECT * FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC",
    format_spec="For each optimization: title, explanation, SQL example, expected improvement."
)
 
result = ask(prompt, max_tokens=600)
display(Markdown(result))

### Optimization 1: Create an Index on Status and Created_at

**Explanation:**  
Creating a composite index on the `status` and `created_at` columns can significantly speed up the query. Indexes help the database quickly locate rows that match the specified criteria without scanning the entire table. When combined, these two columns can efficiently filter the results.

**SQL Example:**  
```sql
CREATE INDEX idx_orders_status_created_at ON orders (status, created_at);
```

**Expected Improvement:**  
With this index in place, the database can quickly find rows where `status = 'pending'` and `created_at > NOW() - INTERVAL '7 days'`, which can vastly reduce the number of rows scanned and improve query performance by up to 80% or more, depending on data distribution.

---

### Optimization 2: Limit the Selected Columns

**Explanation:**  
Instead of selecting all columns with `SELECT *`, it's more efficient to specify only the required columns. This reduces the amount of data transferred and processed, which can lead to faster query execution and less memory usage.

**SQL Example:**  
```sql
SELECT id, status, created_at, total 
FROM orders 
WHERE status='pending' 
  AND created_at > NOW() - INTERVAL '7 days' 
ORDER BY total DESC;
```

**Expected Improvement:**  
By limiting the selected columns, you can reduce the amount of data processed and transferred, which could improve performance by approximately 30%-50%, especially if the `orders` table has many columns or large data types (e.g., text, blobs).

---

### Optimization 3: Use a Materialized View for Frequently Accessed Data

**Explanation:**  
If this query is run frequently, creating a materialized view that precomputes the results can greatly enhance performance. A materialized view stores the result set of the query physically, allowing for much faster access compared to executing the original query against the base table.

**SQL Example:**  
```sql
CREATE MATERIALIZED VIEW mv_pending_orders AS 
SELECT id, status, created_at, total 
FROM orders 
WHERE status='pending' 
  AND created_at > NOW() - INTERVAL '7 days' 
ORDER BY total DESC;

-- To refresh the materialized view periodically
REFRESH MATERIALIZED VIEW mv_pending_orders;
```

**Expected Improvement:**  
Accessing data from a materialized view can reduce query times dramatically, potentially improving performance by 90% or more for repeated queries, as it avoids the overhead of scanning the entire `orders` table each time. However, it will require periodic refreshes to keep the data up to date.

--- 

By implementing these optimizations, you can significantly enhance the performance of your query on the `orders` table, making it faster and more efficient.

---
## 4. Negative Constraints

Telling the model what **NOT** to do is as important as what to do:

In [6]:
prompt = create_pctf_prompt(
    persona="You are a senior developer mentoring a junior",
    context="A junior developer submitted a pull request with this code.",
    task="""Review thw code:
    ```python
    data = []
    for i in range(len(items)):
    if items[i]['active'] == True:
        data.append(items[i]['name'].upper())
    ```""",
    format_spec="""Provide feedback. DO NOT:
    - Be condescending or harsh
    - Rewrite the entire function
    - Use jargon without explanation

    DO:
    - Explain WHY each change improves the code
    - Show improved version
    - Encourage what they did right
    """
)

In [7]:
result = ask(prompt, max_tokens=1000)
display(Markdown(result))

Thank you for your submission! It's great to see you working on this code. You've made a good start by filtering and transforming the data based on the 'active' status. Let's go through some suggestions for improvement.

### Feedback on Your Code

1. **Use of List Comprehension**:
   Your current approach uses a traditional for-loop to iterate through the items. While that works, Python provides a more concise way to handle this with list comprehensions. Using a list comprehension can make your code cleaner and easier to read. 

   **Improvement**: A list comprehension combines the looping and filtering into one line, which can be more efficient and visually clear.

2. **Checking Boolean Values**:
   The condition `if items[i]['active'] == True:` can be simplified. In Python, you can directly use the boolean value without the comparison. If `items[i]['active']` is `True`, it will evaluate as such in the condition. This makes the code a bit cleaner.

   **Improvement**: Using `if items[i]['active']:` is more idiomatic in Python.

3. **Variable Naming**:
   While `data` is a generic name, consider using a more descriptive name that reflects what the list contains. This makes your code more understandable to others who may read it later.

   **Improvement**: A name like `active_item_names` makes it clear that this list holds the names of active items.

### Improved Version
Here’s how your code could look with these suggestions applied:

```python
active_item_names = [item['name'].upper() for item in items if item['active']]
```

### Summary
- The use of list comprehension simplifies the loop and makes the intention of the code clearer.
- Simplifying the boolean check improves readability and follows Python conventions.
- A more descriptive variable name enhances code clarity for anyone reading it.

Overall, you did a great job identifying what you wanted to achieve. Keep up the good work, and consider these suggestions to refine your coding style further! If you have any questions about these changes, feel free to ask.

---
## 5. Exercise — Build Your Own PCTF Prompt 🏋️

In [8]:

prompt = create_pctf_prompt(
    persona="You are senior developer leading the development of a corporate application. Give clear and concise explanations that are easily comprehended by those with less technical knowledge.",
    context="Context: We need to explain the core concept of our app, which is Machine Learning, to non technical stakeholders who we want to invest in our application.",
    task="Explain what Machine Learning is to non technical stakeholders.",
    format_spec="""Include:
    - Simple definition in one sentence.
    - Main characteristics and advantages in bullet points.
    - Offer an allegory so the concept of Machine Learning is more easily understood.
    - Real life example usage."""
)
 
result = ask(prompt, max_tokens=600)
display(Markdown(result))

### Simple Definition
Machine Learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks without being explicitly programmed.

### Main Characteristics and Advantages
- **Data-Driven:** Machine Learning models learn patterns and insights from large sets of data.
- **Adaptability:** These models can adapt to new information, improving their accuracy over time.
- **Automation:** Machine Learning can automate repetitive tasks, freeing up human resources for more complex work.
- **Predictive Analytics:** It can predict outcomes and trends based on historical data, helping businesses make informed decisions.
- **Personalization:** By analyzing user behavior, Machine Learning can tailor experiences and recommendations to individual preferences.

### Allegory
Imagine teaching a child how to recognize different types of fruit. Instead of giving them a strict set of rules, you show them many examples of apples, bananas, and oranges. Over time, the child learns to identify these fruits based on their characteristics, like color and shape, without needing to memorize the rules. Similarly, Machine Learning involves feeding a computer lots of data so it can learn to recognize patterns and make decisions on its own.

### Real-Life Example Usage
A common use of Machine Learning is in online shopping platforms. These sites analyze your browsing and purchasing history to suggest products you might like. For instance, if you frequently buy books about cooking, the site might recommend new cookbooks or kitchen gadgets, enhancing your shopping experience and increasing sales for the retailer.

---
## Key Takeaways 📝

| Component | Tips |
|-----------|------|
| **Persona** | Be specific about expertise level and domain |
| **Context** | Include tech stack, constraints, audience |
| **Task** | One clear, specific action verb |
| **Format** | Tables, lists, JSON — be explicit |
| **Negative constraints** | Say what NOT to do to avoid common pitfalls |
| **Templates** | Build reusable functions for common tasks |

---
**Next:** `05_prompt_evaluation.ipynb` — Measure and compare prompt effectiveness